# 实验 5 拓展：现成模型手势识别

使用 Google MediaPipe 官方预训练模型，无需采集图片或训练。按报告模板设计五种实际手形：**One** 食指、**Two** 食指和中指、**Three** 食指中指无名指、**OK** 拇指食指成环、**Five** 五指张开。官方模型直接分类 One、Two、Five；Three 和 OK 由官方模型的 21 个手部关键点按几何规则判断。画面显示这五个英文标签。识别分数对 One、Two、Five 是模型分类分数，对 Three、OK 是几何余量映射分数，后者并非概率。点击“启动实时画面”后持续取帧并更新识别结果；点击“停止并释放摄像头”后关闭设备。截图和测试记录使用按钮点击时最近的一帧真实画面。

官方说明：[https://developers.google.com/edge/mediapipe/solutions/vision/gesture_recognizer/python](https://developers.google.com/edge/mediapipe/solutions/vision/gesture_recognizer/python)；模型来源：[https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task](https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task)。

In [1]:
import time
from pathlib import Path
import cv2
import numpy as np
import mediapipe as mp
import math

目录 = Path('/home/jetson/Notebook/5.目标识别')
模型路径 = 目录 / 'models/gesture_recognizer.task'
相机号 = 0
最低置信度 = 0.70
手势中文 = {'One': '食指竖起', 'Two': '食指和中指竖起', 'Three': '食指、中指和无名指竖起', 'OK': '拇指与食指成环', 'Five': '五指张开'}
有效手势 = tuple(手势中文)
伸直阈值 = 1.85
弯曲阈值 = 1.55
OK距离阈值 = 0.42

def 创建识别器(模式):
    """用已下载的官方模型创建 MediaPipe 识别器。"""
    if not 模型路径.is_file():
        raise FileNotFoundError('缺少手势模型：' + str(模型路径))
    选项 = mp.tasks.vision.GestureRecognizerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(模型路径)),
        running_mode=模式,
        num_hands=1,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return mp.tasks.vision.GestureRecognizer.create_from_options(选项)

def 识别画面(图像, 识别器, 时间戳=None):
    """用官方类别识别一指、二指和五指张开，用关键点判断三指与 OK。"""
    彩色 = cv2.cvtColor(图像, cv2.COLOR_BGR2RGB)
    输入图 = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(彩色))
    if 时间戳 is None:
        原始结果 = 识别器.recognize(输入图)
    else:
        原始结果 = 识别器.recognize_for_video(输入图, 时间戳)
    官方类别, 官方分数 = '无手', 0.0
    if 原始结果.gestures and 原始结果.gestures[0]:
        首位 = 原始结果.gestures[0][0]
        官方类别, 官方分数 = 首位.category_name, float(首位.score)
    类别, 分数, 分数来源 = '未定义手势', 0.0, '无'
    if not 原始结果.hand_landmarks:
        类别 = '无手'
    else:
        点 = 原始结果.hand_landmarks[0]
        宽高比 = 图像.shape[1] / 图像.shape[0]
        def 距离(甲, 乙):
            return math.hypot((点[甲].x - 点[乙].x) * 宽高比, 点[甲].y - 点[乙].y)
        比值 = [距离(尖端, 根部) / max(距离(近端, 根部), 1e-6) for 根部, 近端, 尖端 in ((5,6,8),(9,10,12),(13,14,16),(17,18,20))]
        环距 = 距离(4, 8) / max(距离(5, 17), 1e-6)
        if 环距 <= OK距离阈值 and all(比值[i] >= 伸直阈值 for i in (1,2,3)):
            余量 = min(OK距离阈值-环距, *(比值[i]-伸直阈值 for i in (1,2,3)))
            类别, 分数, 分数来源 = 'OK', min(0.99, 0.70 + max(0.0, 余量)), '关键点几何余量'
        elif all(比值[i] >= 伸直阈值 for i in (0,1,2)) and 比值[3] <= 弯曲阈值:
            余量 = min(*(比值[i]-伸直阈值 for i in (0,1,2)), 弯曲阈值-比值[3])
            类别, 分数, 分数来源 = 'Three', min(0.99, 0.70 + max(0.0, 余量)), '关键点几何余量'
        elif 官方类别 in ('Pointing_Up', 'Victory', 'Open_Palm') and (官方类别 != 'Open_Palm' or 环距 > OK距离阈值):
            类别 = {'Pointing_Up':'One', 'Victory':'Two', 'Open_Palm':'Five'}[官方类别]
            分数, 分数来源 = 官方分数, '模型分类分数'
    通过 = 类别 in 有效手势 and 分数 >= 最低置信度 and len(原始结果.hand_landmarks) == 1
    return {'类别': 类别, '置信度': 分数, '通过': 通过, '分数来源': 分数来源, '官方类别': 官方类别, '官方分数': 官方分数, '关键点': 原始结果.hand_landmarks}

def 标注画面(图像, 结果):
    """在真实画面上绘制模型手部关键点和类别。"""
    标注 = 图像.copy()
    高, 宽 = 标注.shape[:2]
    for 手 in 结果['关键点']:
        for 点序号甲, 点序号乙 in mp.solutions.hands.HAND_CONNECTIONS:
            甲, 乙 = 手[点序号甲], 手[点序号乙]
            cv2.line(标注, (int(甲.x*宽), int(甲.y*高)), (int(乙.x*宽), int(乙.y*高)), (0, 255, 0), 2)
        for 点 in 手:
            cv2.circle(标注, (int(点.x*宽), int(点.y*高)), 3, (0, 0, 255), -1)
    英文类别 = {'无手': 'No_Hand', '未定义手势': 'Unknown'}.get(结果['类别'], 结果['类别'])
    if not 英文类别.isascii():
        英文类别 = 'Unknown'
    文字 = '{} {:.2f}'.format(英文类别, 结果['置信度'])
    颜色 = (0, 255, 0) if 结果['通过'] else (0, 255, 255)
    cv2.putText(标注, 文字, (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, 颜色, 2)
    return 标注

print('模型文件已找到：{}；可识别手势：{}'.format(模型路径.name, '、'.join(手势中文[x] for x in 有效手势)))


2026-10-26 13:44:02.699034: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.10.2
2026-10-26 13:44:12.489517: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.10.2


模型文件已找到：gesture_recognizer.task；可识别手势：食指竖起、食指和中指竖起、食指、中指和无名指竖起、拇指与食指成环、五指张开


## 离线自检

使用本机旧手势图片检查模型接口。旧图像不是本次实验数据。

In [2]:
from pathlib import Path
样本目录 = Path('/home/jetson/nvdli-data/classification/fingers_A')
with 创建识别器(mp.tasks.vision.RunningMode.IMAGE) as 识别器:
    for 序号 in (1, 2):
        旧样本 = sorted((样本目录 / str(序号)).glob('*.jpg'))
        if not 旧样本:
            print('旧样本缺失：', 序号)
            continue
        图像 = cv2.imread(str(旧样本[0]))
        结果 = 识别画面(图像, 识别器)
        print('旧样本 {}：{}，识别分数 {:.3f}；仅验证模型接口，不计入本次实验。'.format(序号, 结果['类别'], 结果['置信度']))


W0000 00:00:1792993455.329628   26768 gesture_recognizer_graph.cc:129] Hand Gesture Recognizer contains CPU only ops. Sets HandGestureRecognizerGraph acceleration to Xnnpack.
I0000 00:00:1792993455.362822   26768 hand_gesture_recognizer_graph.cc:250] Custom gesture classifier is not defined.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


旧样本 1：One，识别分数 0.751；仅验证模型接口，不计入本次实验。
旧样本 2：Two，识别分数 0.779；仅验证模型接口，不计入本次实验。


## 实时识别与记录

依次做 One、Two、Three、OK、Five。点击“启动实时画面”后，观察连续更新的标签和手部关键点；需要取证时点击“保存当前截图”或填写真实手势后点击“记录一次真实测试”。画面左上角使用英文标签，中文说明在画面下方。实验结束务必点击“停止并释放摄像头”。Three 和 OK 的几何阈值在第一格调整，须记录实际使用值。

In [3]:
import csv
import threading
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display

try:
    if 采集线程 is not None and 采集线程.is_alive():
        停止事件.set()
        采集线程.join(timeout=5)
        if 采集线程.is_alive():
            raise RuntimeError('旧实时循环仍在运行；请先停止或重启内核。')
except NameError:
    pass

证据目录 = 目录 / 'gesture_evidence'
记录路径 = 目录 / 'week5_gesture_records.csv'
停止事件 = threading.Event()
画面锁 = threading.Lock()
采集线程 = None
当前原图 = None
当前结果 = None
当前帧时间 = None

启动按钮 = widgets.Button(description='启动实时画面')
停止按钮 = widgets.Button(description='停止并释放摄像头')
保存按钮 = widgets.Button(description='保存当前截图')
记录按钮 = widgets.Button(description='记录一次真实测试')
实际手势框 = widgets.Dropdown(options=[(手势中文[x], x) for x in 有效手势], description='实际手势')
备注框 = widgets.Text(description='现场备注')
画面框 = widgets.Image(format='jpeg', width=640)
状态框 = widgets.HTML(value='等待启动。')

def 读取当前帧():
    """取得同一帧的原图和识别结果，供截图与记录使用。"""
    with 画面锁:
        if 当前原图 is None:
            return None
        return 当前原图.copy(), 当前结果, 当前帧时间

def 实时循环():
    """持续读取和识别；退出时关闭模型与摄像头。"""
    global 当前原图, 当前结果, 当前帧时间
    摄像头 = cv2.VideoCapture(相机号)
    识别器 = None
    try:
        if not 摄像头.isOpened():
            raise RuntimeError('摄像头无法打开，请检查其他 notebook 是否占用。')
        识别器 = 创建识别器(mp.tasks.vision.RunningMode.VIDEO)
        上次时间戳 = -1
        while not 停止事件.is_set():
            成功, 图像 = 摄像头.read()
            if not 成功:
                raise RuntimeError('摄像头取帧失败；实时识别已经停止。')
            时间戳 = max(int(time.monotonic() * 1000), 上次时间戳 + 1)
            上次时间戳 = 时间戳
            结果 = 识别画面(图像, 识别器, 时间戳)
            标注 = 标注画面(图像, 结果)
            成功编码, 编码 = cv2.imencode('.jpg', 标注)
            with 画面锁:
                当前原图 = 图像.copy()
                当前结果 = 结果
                当前帧时间 = datetime.now()
            if 成功编码:
                画面框.value = 编码.tobytes()
            状态框.value = '实时识别：{}（{}）；分数：{:.3f}；来源：{}；通过阈值：{}。'.format(结果['类别'], 手势中文.get(结果['类别'], '其他'), 结果['置信度'], 结果['分数来源'], 结果['通过'])
            停止事件.wait(0.12)
    except Exception as 错误:
        状态框.value = '实时识别出错：' + str(错误)
    finally:
        if 识别器 is not None:
            识别器.close()
        摄像头.release()
        if 停止事件.is_set():
            状态框.value = '已停止并释放摄像头。'

def 启动(_):
    global 采集线程, 当前原图, 当前结果, 当前帧时间
    if 采集线程 is not None and 采集线程.is_alive():
        状态框.value = '实时画面已经运行。'
        return
    with 画面锁:
        当前原图 = None
        当前结果 = None
        当前帧时间 = None
    停止事件.clear()
    采集线程 = threading.Thread(target=实时循环, daemon=True)
    采集线程.start()
    状态框.value = '正在启动实时画面……'

def 停止(_):
    停止事件.set()
    状态框.value = '已请求停止；当前取帧结束后释放摄像头。'

def 保存(_):
    帧 = 读取当前帧()
    if 帧 is None:
        状态框.value = '尚无真实画面，请先启动。'
        return
    原图, 结果, 帧时间 = 帧
    证据目录.mkdir(exist_ok=True)
    时间 = 帧时间.strftime('%Y%m%d_%H%M%S_%f')
    原图名, 结果名 = 时间 + '_raw.png', 时间 + '_gesture.png'
    cv2.imwrite(str(证据目录 / 原图名), 原图)
    cv2.imwrite(str(证据目录 / 结果名), 标注画面(原图, 结果))
    状态框.value = '已保存真实截图：{}、{}'.format(原图名, 结果名)

def 记录(_):
    帧 = 读取当前帧()
    if 帧 is None:
        状态框.value = '尚无真实画面，请先启动。'
        return
    原图, 结果, 帧时间 = 帧
    证据目录.mkdir(exist_ok=True)
    时间 = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    图片名 = 时间 + '_week5_trial.png'
    cv2.imwrite(str(证据目录 / 图片名), 标注画面(原图, 结果))
    新文件 = not 记录路径.exists()
    with 记录路径.open('a', newline='', encoding='utf-8-sig') as 文件:
        表格 = csv.writer(文件)
        if 新文件:
            表格.writerow(['记录时间', '画面时间', '实际手势', '识别类别', '识别分数', '分数来源', '是否正确', '备注', '截图'])
        正确 = int(结果['类别'] == 实际手势框.value and 结果['通过'])
        表格.writerow([时间, 帧时间.isoformat(timespec='milliseconds'), 实际手势框.value, 结果['类别'], '{:.4f}'.format(结果['置信度']), 结果['分数来源'], 正确, 备注框.value, 图片名])
    状态框.value = '真实测试已记录；继续摆出下一次手势即可。'

启动按钮.on_click(启动)
停止按钮.on_click(停止)
保存按钮.on_click(保存)
记录按钮.on_click(记录)
display(widgets.VBox([widgets.HBox([启动按钮, 停止按钮, 保存按钮]), 画面框, widgets.HBox([实际手势框, 备注框, 记录按钮]), 状态框]))